In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train=pd.read_csv('../input/forest-cover-type-prediction/train.csv')
train.head()

In [ ]:
abs(train.corr()['Cover_Type'].sort_values(ascending=False))

In [ ]:
x=train.drop(['Id','Cover_Type'],axis=1)
y=train['Cover_Type']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix,precision_score,recall_score,f1_score

In [ ]:
def classify(x, y):
    k=KNeighborsClassifier()
    svc=SVC()
    d=DecisionTreeClassifier()
    log=LogisticRegression()
    rf=RandomForestClassifier()
    
    #random_state is the seed to create reproducible output
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=.2,random_state=42)
    
    algos=[k,svc,d,log,rf]
    algos_name=['KNeigbors','SVC','DecisionTree','LogisticRegr','RandomForest']
    
    accuracy = []
    precision = []
    recall = []
    f1 = []
   
    result=pd.DataFrame(columns=['AccuracyScore','PrecisionScore','RecallScore','f1_Score'],index=algos_name)
    
    for i in algos:
        
        predict=i.fit(x_train,y_train).predict(x_test)
        
        accuracy.append(accuracy_score(y_test,predict))
        precision.append(precision_score(y_test,predict,average='micro'))
        recall.append(recall_score(y_test,predict,average='micro'))
        f1.append(f1_score(y_test,predict,average='micro'))
        

    
    
    # correctness of predictions
    result.AccuracyScore=accuracy
    # The precision is the ratio tp / (tp + fp) where tp is the number of true positives and 
    # fp the number of false positives. The precision is intuitively the ability of the 
    # classifier not to label as positive a sample that is negative
    result.PrecisionScore=precision
    # The recall is the ratio tp / (tp + fn) where tp is the number of true positives and fn 
    # the number of false negatives. The recall is intuitively the ability of the classifier 
    # to find all the positive samples.
    result.RecallScore=recall
    # The F1 score can be interpreted as a harmonic mean of the precision and recall,
    # F1 = 2 * (precision * recall) / (precision + recall)
    result.f1_Score=f1
    
    # more metrics: https://scikit-learn.org/stable/modules/model_evaluation.html
    
    
    ret = result.sort_values('f1_Score',ascending=False)
    bestModelName = ret.iloc[0].name
    bestModel = algos[algos_name.index(bestModelName)]
    return (ret, bestModel)
    

In [ ]:
results = classify(x,y)
results[0]

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y, random_state=42)
model = results[1]
model.fit(x_train, y_train)
model.score(x_test,y_test)

In [ ]:
test = pd.read_csv('../input/forest-cover-type-prediction/test.csv')
test.head()

In [ ]:
# model.fit() vs model.predict(): fit is used when our df has the response variable,
# predict is used when that is missing (like in test data)
predict=model.predict(test.drop(labels=['Id'],axis=1))
submission= pd.DataFrame(data = predict,columns = ['Cover_Type'])
submission.head()

In [ ]:
submission['Id'] =test['Id']
submission.set_index('Id',inplace=True)
submission.to_csv('submission.csv')

# HOMEWORK
## 1. Preprocess your data
Modify your train data, using strategies we went over in the first meeting <br>
You can look at other notebooks around Kaggle for inspiration<br>
## 2. Use more models
List of all Scikit-Learn models: https://scikit-learn.org/stable/supervised_learning.html <br>
Also try lazy model training libraries like AutoML and LazyPredict (most likely will need to pip install these)<br>
... Or any other libraries, google!<br>
In some of the models, additional parameters can be passed in too (such as number of estimators for RandomForest). See if those make a difference
## 3. Get a score greater than 0.75
May or may not have a prize for the highest score 👀